In [ ]:
# cv_summary_all_experiments.py
#
# Purpose: Summarize 5-fold CV results for all experiments
# Output: printed table per experiment with mean and std
# Note: skips missing or empty results.csv files

import pandas as pd
import os

EXPERIMENTS = {
    "Summer COCO v1":        r"C:\Master Physische Geographie\Canada\Masterarbeit\results\summer_coco_2cls",
    "Fall COCO v1":          r"C:\Master Physische Geographie\Canada\Masterarbeit\results\fall_coco_2cls",
    "Summer P2 v2":          r"C:\Master Physische Geographie\Canada\Masterarbeit\results\summer_p2_2cls_v2",
    "Fall P2 v2":            r"C:\Master Physische Geographie\Canada\Masterarbeit\results\fall_p2_2cls_v2",
    "Summer v3 1280px":      r"C:\Master Physische Geographie\Canada\Masterarbeit\results\summer_coco_2cls_v3",
    "Fall v3 1280px":        r"C:\Master Physische Geographie\Canada\Masterarbeit\results\fall_coco_2cls_v3",
    "Summer v4 augmented":   r"C:\Master Physische Geographie\Canada\Masterarbeit\results\summer_coco_2cls_v4",
    "Fall v4 augmented":     r"C:\Master Physische Geographie\Canada\Masterarbeit\results\fall_coco_2cls_v4",
    "Summer v5 augmented":   r"C:\Master Physische Geographie\Canada\Masterarbeit\results\summer_coco_2cls_v5",
    "Fall v5 augmented":     r"C:\Master Physische Geographie\Canada\Masterarbeit\results\fall_coco_2cls_v5",
    "Summer v6 200ep":       r"C:\Master Physische Geographie\Canada\Masterarbeit\results\summer_coco_2cls_v6",
    "Fall v6 200ep":         r"C:\Master Physische Geographie\Canada\Masterarbeit\results\fall_coco_2cls_v6_correct",
    "Summer RT-DETR v7":     r"C:\Master Physische Geographie\Canada\Masterarbeit\results\summer_rtdetr_2cls_v7",
    "Fall RT-DETR v7":       r"C:\Master Physische Geographie\Canada\Masterarbeit\results\fall_rtdetr_2cls_v7",
    "Summer RT-DETR v8":     r"C:\Master Physische Geographie\Canada\Masterarbeit\results\summer_rtdetr_2cls_v8",
    "Fall RT-DETR v8":       r"C:\Master Physische Geographie\Canada\Masterarbeit\results\fall_rtdetr_2cls_v8",
    "Summer RT-DETR v9":     r"C:\Master Physische Geographie\Canada\Masterarbeit\results\summer_rtdetr_2cls_v9_full",
    "Fall RT-DETR v9":       r"C:\Master Physische Geographie\Canada\Masterarbeit\results\fall_rtdetr_2cls_v9",
    "Summer RT-DETR v10":    r"C:\Master Physische Geographie\Canada\Masterarbeit\results\summer_rtdetr_2cls_v10_full",
    "Fall RT-DETR v10":      r"C:\Master Physische Geographie\Canada\Masterarbeit\results\fall_rtdetr_2cls_v10_full",
    "Summer RT-DETR v11 3cls": r"C:\Master Physische Geographie\Canada\Masterarbeit\results\summer_rtdetr_3cls_v11",
    "Fall RT-DETR v11 3cls":   r"C:\Master Physische Geographie\Canada\Masterarbeit\results\fall_rtdetr_3cls_v11",
    "Summer RT-DETR v12 iou0.6": r"C:\Master Physische Geographie\Canada\Masterarbeit\results\summer_rtdetr_2cls_v12",
    "Fall RT-DETR v12 iou0.6":   r"C:\Master Physische Geographie\Canada\Masterarbeit\results\fall_rtdetr_2cls_v12",
    "Summer YOLOv8 v13 3cls":    r"C:\Master Physische Geographie\Canada\Masterarbeit\results\summer_yolo_3cls_v13",
    "Fall YOLOv8 v13 3cls":      r"C:\Master Physische Geographie\Canada\Masterarbeit\results\fall_yolo_3cls_v13",
    "Summer YOLOv9 v14":     r"C:\Master Physische Geographie\Canada\Masterarbeit\results\summer_yolov9_2cls_v14",
    "Fall YOLOv9 v14":       r"C:\Master Physische Geographie\Canada\Masterarbeit\results\fall_yolov9_2cls_v14",
    "Summer RT-DETR-X v15":  r"C:\Master Physische Geographie\Canada\Masterarbeit\results\summer_rtdetrx_2cls_v15",
    "Fall RT-DETR-X v15":    r"C:\Master Physische Geographie\Canada\Masterarbeit\results\fall_rtdetrx_2cls_v15",
    "Summer YOLOv10 v16":    r"C:\Master Physische Geographie\Canada\Masterarbeit\results\summer_yolov10_2cls_v16",
    "Fall YOLOv10 v16":      r"C:\Master Physische Geographie\Canada\Masterarbeit\results\fall_yolov10_2cls_v16",
    "Summer YOLOv11 v18":    r"C:\Master Physische Geographie\Canada\Masterarbeit\results\summer_yolov11_2cls_v18",
    "Fall YOLOv11 v18":      r"C:\Master Physische Geographie\Canada\Masterarbeit\results\fall_yolov11_2cls_v18",
    "Summer BiFPN v19":      r"C:\Master Physische Geographie\Canada\Masterarbeit\results\summer_yolo_bifpn_2cls_v19",
    "Fall BiFPN v19": r"C:\Master Physische Geographie\Canada\Masterarbeit\results\fall_yolo_bifpn_2cls_v19",
    "Summer MPDIoU v20":     r"C:\Master Physische Geographie\Canada\Masterarbeit\results\summer_yolo_mpdiou_2cls_v20",
    "Fall MPDIoU v20":       r"C:\Master Physische Geographie\Canada\Masterarbeit\results\fall_yolo_mpdiou_2cls_v20",
    "Summer RT-DETR v17c dropout+scale": r"C:\Master Physische Geographie\Canada\Masterarbeit\results\summer_rtdetr_2cls_v17c",
    "Fall RT-DETR v17c dropout+scale":   r"C:\Master Physische Geographie\Canada\Masterarbeit\results\fall_rtdetr_2cls_v17c",
}

summary_rows = []

for exp_name, results_base in EXPERIMENTS.items():
    rows = []
    for fold in range(1, 6):
        csv_path = os.path.join(results_base, f"fold{fold}", "results.csv")
        if not os.path.exists(csv_path):
            print(f"WARNING: not found — {csv_path}")
            continue
        if os.path.getsize(csv_path) == 0:
            print(f"WARNING: empty file — {csv_path}")
            continue
        try:
            df = pd.read_csv(csv_path, header=0)
            df.columns = [c.strip() for c in df.columns]
            best = df.loc[df["metrics/mAP50(B)"].idxmax()]
            rows.append({
                "fold":       fold,
                "best_epoch": int(best.iloc[0]),
                "precision":  best["metrics/precision(B)"],
                "recall":     best["metrics/recall(B)"],
                "mAP50":      best["metrics/mAP50(B)"],
                "mAP50_95":   best["metrics/mAP50-95(B)"],
            })
        except Exception as e:
            print(f"WARNING: error reading {csv_path} — {e}")
            continue

    if not rows:
        print(f"SKIPPING {exp_name} — no valid folds")
        continue

    fold_df = pd.DataFrame(rows)
    mean = fold_df[["precision","recall","mAP50","mAP50_95"]].mean()
    std  = fold_df[["precision","recall","mAP50","mAP50_95"]].std()
    n_folds = len(rows)

    print(f"\n{'='*60}")
    print(f"Experiment: {exp_name} ({n_folds} folds)")
    print(f"{'='*60}")
    print(fold_df.to_string(index=False))
    print(f"\nMean:  P={mean['precision']:.4f}  R={mean['recall']:.4f}  mAP50={mean['mAP50']:.4f}  mAP50-95={mean['mAP50_95']:.4f}")
    print(f"Std:   P={std['precision']:.4f}   R={std['recall']:.4f}   mAP50={std['mAP50']:.4f}   mAP50-95={std['mAP50_95']:.4f}")

    summary_rows.append({
        "experiment":  exp_name,
        "n_folds":     n_folds,
        "precision":   round(mean["precision"], 4),
        "recall":      round(mean["recall"], 4),
        "mAP50":       round(mean["mAP50"], 4),
        "mAP50_95":    round(mean["mAP50_95"], 4),
        "mAP50_std":   round(std["mAP50"], 4),
    })

print(f"\n{'='*60}")
print("OVERALL COMPARISON")
print(f"{'='*60}")
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))


Experiment: Summer COCO v1 (5 folds)
 fold  best_epoch  precision  recall   mAP50  mAP50_95
    1          62    0.51388 0.52597 0.49082   0.20229
    2          45    0.54302 0.57098 0.56297   0.24447
    3          63    0.59071 0.62298 0.59520   0.22914
    4          90    0.63964 0.62131 0.63572   0.26966
    5          55    0.60092 0.57127 0.60236   0.26382

Mean:  P=0.5776  R=0.5825  mAP50=0.5774  mAP50-95=0.2419
Std:   P=0.0495   R=0.0406   mAP50=0.0549   mAP50-95=0.0273

Experiment: Fall COCO v1 (5 folds)
 fold  best_epoch  precision  recall   mAP50  mAP50_95
    1          40    0.35172 0.22464 0.19080   0.05633
    2          41    0.29740 0.26711 0.20089   0.05726
    3          48    0.31768 0.26564 0.19897   0.05681
    4          81    0.37564 0.30411 0.26597   0.07495
    5          61    0.42251 0.31469 0.28763   0.09206

Mean:  P=0.3530  R=0.2752  mAP50=0.2289  mAP50-95=0.0675
Std:   P=0.0492   R=0.0357   mAP50=0.0446   mAP50-95=0.0158

Experiment: Summer P2 v2 (5 f

In [ ]:
# check if any results.csv or log files mention the version
import glob, os

for f in glob.glob(r"C:\Master Physische Geographie\Canada\Masterarbeit\results\**\*.yaml", recursive=True):
    with open(f) as fp:
        content = fp.read()
    if "ultralytics" in content.lower() or "torch" in content.lower():
        print(f)
        print(content[:500])
        print()
        break

18895
